# Week 3 · Class 1 — SQL Fundamentals
**AI Training Program · Phase 1: Python Foundations**

---

## What you'll learn
- What databases are and why they matter
- How to create and connect to a SQLite database in Python
- SQL data types and table structure
- Core CRUD operations: `CREATE`, `INSERT`, `SELECT`, `UPDATE`, `DELETE`
- Using `sqlite3` and `pandas` together

---

## Why SQL?

Almost every real-world application stores data in a relational database. As an AI practitioner you'll constantly be:
- Pulling training data from databases
- Logging model predictions and metrics
- Querying business data to understand context

SQL is the universal language for all of this.

## 1. Setting Up — `sqlite3`

Python ships with `sqlite3` built in. SQLite stores an entire database in a single `.db` file — perfect for learning and small projects.

In [1]:
import sqlite3
import pandas as pd

# Connect (creates the file if it doesn't exist)
conn = sqlite3.connect("school.db")
cursor = conn.cursor()

print("Connected to school.db ✓")
print(f"SQLite version: {sqlite3.sqlite_version}")

Connected to school.db ✓
SQLite version: 3.45.3


In [ ]:
#file_paths = 'S3:bucket/bucket_name'

## 2. SQL Data Types

| SQL Type | Stores | Example |
|---|---|---|
| `INTEGER` | Whole numbers | `42`, `-7` |
| `REAL` | Floating point | `3.14`, `98.6` |
| `TEXT` | Strings | `'Alice'`, `'Mumbai'` |
| `BLOB` | Raw binary data | Images, files |
| `NULL` | Missing / unknown | `NULL` |

> SQLite is loosely typed — it calls this **type affinity**. Other databases (PostgreSQL, MySQL) are stricter.

## 3. `CREATE TABLE` — Defining Structure

A table is like a spreadsheet with fixed columns. You define the columns and their types once.

In [2]:
# Drop tables if re-running this notebook
cursor.execute("DROP TABLE IF EXISTS students")
cursor.execute("DROP TABLE IF EXISTS courses")
cursor.execute("DROP TABLE IF EXISTS enrollments")

# Create the students table -> table_name = students
cursor.execute("""
    CREATE TABLE students (
        student_id   INTEGER PRIMARY KEY AUTOINCREMENT,
        name         TEXT    NOT NULL,
        email        TEXT    UNIQUE NOT NULL,
        age          INTEGER CHECK(age >= 16 AND age <= 100),
        city         TEXT    DEFAULT 'Mumbai',
        enrolled_on  TEXT    DEFAULT (date('now'))
    )
""")

# Create the courses table
cursor.execute("""
    CREATE TABLE courses (
        course_id    INTEGER PRIMARY KEY AUTOINCREMENT,
        title        TEXT    NOT NULL,
        instructor   TEXT    NOT NULL,
        credits      INTEGER DEFAULT 3
    )
""")

conn.commit()
print("Tables created ✓")

Tables created ✓


### Key constraints explained

| Constraint | Meaning |
|---|---|
| `PRIMARY KEY` | Uniquely identifies every row |
| `AUTOINCREMENT` | DB assigns the next integer automatically |
| `NOT NULL` | Column cannot be empty |
| `UNIQUE` | No two rows can have the same value |
| `CHECK(condition)` | Rejects rows where condition is false |
| `DEFAULT value` | Used when no value is provided |

## 4. `INSERT INTO` — Adding Data

There are two styles. Always use **parameterised queries** (the `?` placeholders) when inserting user-supplied data — it prevents SQL injection.

In [3]:
# --- Style 1: Insert one row with a tuple ---
cursor.execute("""
    INSERT INTO students (name, email, age, city)
    VALUES (?, ?, ?, ?)
""", ("Aarav Shah", "aarav@example.com", 24, "Mumbai"))

# --- Style 2: Insert many rows at once ---
more_students = [
    ("Priya Nair",   "priya@example.com",   22, "Pune"),
    ("Rohan Mehta",  "rohan@example.com",   26, "Delhi"),
    ("Sana Khan",    "sana@example.com",    23, "Mumbai"),
    ("Dev Sharma",   "dev@example.com",     28, "Bangalore"),
    ("Anita Bose",   "anita@example.com",   21, "Kolkata"),
    ("Kiran Rao",    "kiran@example.com",   25, "Hyderabad"),
]

cursor.executemany("""
    INSERT INTO students (name, email, age, city)
    VALUES (?, ?, ?, ?)
""", more_students)

conn.commit()
print(f"{cursor.rowcount} rows inserted into students")

6 rows inserted into students


In [ ]:
# Insert courses
courses_data = [
    ("Python Foundations",  "Dr. Mehta",  4),
    ("Machine Learning",    "Dr. Rao",    4),
    ("SQL & Databases",     "Dr. Khan",   3),
    ("Deep Learning",       "Dr. Sharma", 4),
    ("MLOps & Deployment",  "Dr. Nair",   3),
]

cursor.executemany("""
    INSERT INTO courses (title, instructor, credits)
    VALUES (?, ?, ?)
""", courses_data)

conn.commit()
print("Courses inserted ✓")

## 5. `SELECT` — Reading Data

The anatomy of a SELECT statement:

```sql
SELECT  column1, column2   -- what to retrieve  (* = all columns)
FROM    table_name         -- which table
WHERE   condition          -- filter rows (optional)
ORDER BY column DESC       -- sort results (optional)
LIMIT   n;                 -- cap the number of rows (optional)
```

In [ ]:
# Select all columns from students
df = pd.read_sql_query("SELECT * FROM students", conn)
df

In [ ]:
# Select specific columns
df_names = pd.read_sql_query("""
    SELECT name, city, age
    FROM   students
    ORDER BY age DESC
""", conn)
df_names

In [ ]:
# SELECT with computed columns and aliases
df_computed = pd.read_sql_query("""
    SELECT
        name,
        age,
        age + 10             AS age_in_10_years,
        UPPER(city)          AS city_upper,
        LENGTH(name)         AS name_length
    FROM students
    LIMIT 5
""", conn)
df_computed

## 6. `UPDATE` — Modifying Existing Rows

In [ ]:
# Update a single student's city
cursor.execute("""
    UPDATE students
    SET    city = 'Chennai'
    WHERE  name = 'Aarav Shah'
""")
conn.commit()

print(f"Rows affected: {cursor.rowcount}")

# Verify
pd.read_sql_query("SELECT name, city FROM students WHERE name = 'Aarav Shah'", conn)

In [ ]:
# ⚠️  UPDATE without WHERE updates ALL rows — be careful!
# This would set EVERY student's city to 'Unknown':
# cursor.execute("UPDATE students SET city = 'Unknown'")

# Safe pattern: always preview what you'll update first
preview = pd.read_sql_query("""
    SELECT name, age FROM students WHERE age < 23
""", conn)
print("Rows that would be updated:")
print(preview)

## 7. `DELETE` — Removing Rows

In [ ]:
# First: check what we'll delete
print("Before delete:", pd.read_sql_query("SELECT COUNT(*) AS total FROM students", conn).iloc[0,0], "students")

# Insert a test row to delete
cursor.execute("""
    INSERT INTO students (name, email, age, city)
    VALUES ('Test User', 'test@delete.com', 99, 'Nowhere')
""")

# Now delete it
cursor.execute("DELETE FROM students WHERE email = 'test@delete.com'")
conn.commit()

print("After delete:", pd.read_sql_query("SELECT COUNT(*) AS total FROM students", conn).iloc[0,0], "students")

## 8. Inspecting the Schema

In [ ]:
# List all tables in the database
tables = pd.read_sql_query("""
    SELECT name, type FROM sqlite_master
    WHERE  type IN ('table', 'view')
    ORDER BY name
""", conn)
print("Tables in school.db:")
print(tables)

# Inspect column info for a table
print("\nColumns in students table:")
cursor.execute("PRAGMA table_info(students)")
for col in cursor.fetchall():
    print(f"  {col[1]:15s}  type={col[2]:10s}  notnull={col[3]}  default={col[4]}")

## 9. Using `pandas.read_sql_query` vs `cursor.fetchall()`

Two ways to get results back into Python:

In [7]:
# --- Option A: cursor (raw tuples) ---
cursor.execute("SELECT name, age FROM students ORDER BY age LIMIT 3")
rows = cursor.fetchall() # .fetchall(), .fetchone().
print("Raw tuples:", rows)

Raw tuples: [('Anita Bose', 21), ('Priya Nair', 22), ('Sana Khan', 23)]


In [8]:
# --- Option A: cursor (raw tuples) ---
cursor.execute("SELECT name, age FROM students ORDER BY age LIMIT 3")
rows = cursor.fetchall()
print("Raw tuples:", rows)

print()

# --- Option B: pandas (DataFrame) ---
df = pd.read_sql_query("SELECT name, age FROM students ORDER BY age LIMIT 3", conn)
print("DataFrame:")
print(df)

# Prefer pandas when you'll do further analysis
# Prefer cursor for simple lookups or when pandas isn't available

Raw tuples: [('Anita Bose', 21), ('Priya Nair', 22), ('Sana Khan', 23)]

DataFrame:
         name  age
0  Anita Bose   21
1  Priya Nair   22
2   Sana Khan   23


---

## ✏️ Exercises

Complete each task in the cell below it.

**Exercise 1**: Create a `teachers` table with columns: `teacher_id` (PK autoincrement), `name` (text, not null), `subject` (text), `experience_years` (integer, default 1).

In [ ]:
# Your code here


**Exercise 2**: Insert at least 3 teachers into the `teachers` table using `executemany`.

In [ ]:
# Your code here


**Exercise 3**: Select all students from Mumbai, ordered by name alphabetically.

In [ ]:
# Your code here


**Exercise 4**: Update Kiran Rao's age to 27 and verify the change.

In [ ]:
# Your code here


---

## Summary

| Command | Purpose |
|---|---|
| `CREATE TABLE` | Define structure with columns and constraints |
| `INSERT INTO` | Add new rows (use `?` placeholders!) |
| `SELECT` | Read data — choose columns, filter, sort |
| `UPDATE ... SET` | Modify existing rows (always use `WHERE`!) |
| `DELETE FROM` | Remove rows (always use `WHERE`!) |
| `PRAGMA table_info` | Inspect a table's schema |

**Next class**: `WHERE`, comparison operators, `LIKE`, `BETWEEN`, `ORDER BY`, and `LIMIT`.

In [ ]:
# Keep the connection open — next notebooks will reuse school.db
conn.close()
print("school.db saved and connection closed.")